In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pybaseball import statcast_pitcher

PLAYER_ID = 694973
START_DATE = "2025-03-01"
END_DATE = "2025-11-01"

df = statcast_pitcher(START_DATE, END_DATE, PLAYER_ID)

df.head()

In [ ]:
cols = [
    "game_date", "pitch_type", "pitch_name", "release_speed",
    "description", "events", "zone", "plate_x", "plate_z",
    "batter", "stand", "p_throws", "balls", "strikes"
]

skenes = df[cols].copy()
skenes = skenes.dropna(subset=["pitch_type", "release_speed"])

skenes.head()

In [ ]:
pitch_summary = (
    skenes.groupby(["pitch_type", "pitch_name"])
    .agg(
        pitches=("pitch_type", "count"),
        avg_velocity=("release_speed", "mean")
    )
    .reset_index()
)

pitch_summary["usage_rate"] = pitch_summary["pitches"] / pitch_summary["pitches"].sum()
pitch_summary = pitch_summary.sort_values("pitches", ascending=False)

pitch_summary

In [ ]:
swinging_strikes = ["swinging_strike", "swinging_strike_blocked"]

whiff_summary = (
    skenes.assign(is_whiff=skenes["description"].isin(swinging_strikes))
    .groupby(["pitch_type", "pitch_name"])
    .agg(
        pitches=("pitch_type", "count"),
        whiffs=("is_whiff", "sum"),
        avg_velocity=("release_speed", "mean")
    )
    .reset_index()
)

whiff_summary["whiff_rate"] = whiff_summary["whiffs"] / whiff_summary["pitches"]
whiff_summary = whiff_summary.sort_values("whiff_rate", ascending=False)

whiff_summary

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(pitch_summary["pitch_name"], pitch_summary["usage_rate"])
plt.title("Paul Skenes Pitch Usage")
plt.ylabel("Usage Rate")
plt.xlabel("Pitch Type")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../outputs/skenes_pitch_usage.png", dpi=300)
plt.show()

In [ ]:
velocity_summary = pitch_summary.sort_values("avg_velocity", ascending=False)

plt.figure(figsize=(10, 6))
plt.bar(velocity_summary["pitch_name"], velocity_summary["avg_velocity"])
plt.title("Paul Skenes Average Velocity by Pitch")
plt.ylabel("Average Velocity (mph)")
plt.xlabel("Pitch Type")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../outputs/skenes_velocity.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(whiff_summary["pitch_name"], whiff_summary["whiff_rate"])
plt.title("Paul Skenes Whiff Rate by Pitch")
plt.ylabel("Whiff Rate")
plt.xlabel("Pitch Type")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../outputs/skenes_whiff_rate.png", dpi=300)
plt.show()

In [ ]:
strike_descriptions = [
    "called_strike", "swinging_strike", "swinging_strike_blocked",
    "foul", "foul_tip", "foul_bunt", "missed_bunt", "bunt_foul_tip",
    "hit_into_play"
]

strike_summary = (
    skenes.assign(is_strike=skenes["description"].isin(strike_descriptions))
    .groupby(["pitch_type", "pitch_name"])
    .agg(
        pitches=("pitch_type", "count"),
        strikes=("is_strike", "sum"),
        avg_velocity=("release_speed", "mean")
    )
    .reset_index()
)

strike_summary["strike_rate"] = strike_summary["strikes"] / strike_summary["pitches"]
strike_summary = strike_summary.sort_values("strike_rate", ascending=False)

plt.figure(figsize=(10, 6))
plt.bar(strike_summary["pitch_name"], strike_summary["strike_rate"])
plt.title("Paul Skenes Strike Rate by Pitch")
plt.ylabel("Strike Rate")
plt.xlabel("Pitch Type")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../outputs/skenes_strike_rate.png", dpi=300)
plt.show()

strike_summary